# Representation Engineering: Activation Steering & The Linear Representation Hypothesis

> **Topic**: Mechanistic Interpretability, Model Steering, Safety Alignment  
> **Key Breakthrough**: Controlling LLM behavior (truthfulness, refusal, persona) by adding concept vectors directly into residual streams.

---

## 1. The Linear Representation Hypothesis
Neural network representations map high-level conceptual abstractions (honesty, sentiment, toxicity, sycophancy) to linear one-dimensional directional subspaces in the hidden activation space:
$$\mathbf{h}_{\text{concept}} = \mathbf{v}_{\text{direction}} \cdot s + \mathbf{h}_{\text{neutral}}$$

### Contrastive Activation Addition (CAA)
Given positive concept prompts $P_+$ (e.g. "Give a truthful, factual answer") and negative prompts $P_-$ (e.g. "Give an untrue, fabricated answer"):
1. Record hidden activations at layer $l$: $\mathbf{H}_+ = \{\mathbf{h}_{l, i}^+\}$ and $\mathbf{H}_- = \{\mathbf{h}_{l, i}^-\}$.
2. Compute the concept steering vector:
   $$\mathbf{v}_l = \frac{1}{|P_+|} \sum_{i} \mathbf{h}_{l, i}^+ - \frac{1}{|P_-|} \sum_{j} \mathbf{h}_{l, j}^-$$
3. During inference on arbitrary queries, steer model behavior:
   $$\mathbf{h}'_l = \mathbf{h}_l + \alpha \cdot \frac{\mathbf{v}_l}{\|\mathbf{v}_l\|}$$


In [ ]:
import torch

def steer_hidden_state(hidden_state, steering_vector, alpha=1.5):
    """Apply linear activation steering in-place during forward pass."""
    norm_v = steering_vector / (torch.norm(steering_vector) + 1e-8)
    return hidden_state + alpha * norm_v

# Test vector addition
h = torch.randn(1, 10, 4096)
v_truth = torch.randn(4096)
steered_h = steer_hidden_state(h, v_truth, alpha=2.0)
print(f"Original norm: {h.norm():.2f} -> Steered norm: {steered_h.norm():.2f}")
